# QT Cluster analysis

## Select Random?(Higher weight higher chance) Sample for clustering n=100

In [1]:
import mdtraj as md
import numpy as np
import pandas as pd
import sys
import os

In [2]:
tag = "hcp_rdc_3j_theta=16"

In [ ]:
BME_DIR = "bme_reweight"

w_rew = np.load(f"{BME_DIR}/crossvals/crossval_{tag}/weights_results.npy")[0]
w_rew_2 = np.column_stack((np.array(range(20100)), w_rew))
df = pd.DataFrame(w_rew_2)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_2.dat', index=False)

w_rew_3 = w_rew_2[np.argsort(w_rew_2[:,1])]
np.save(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_weights_frames.npy', w_rew_3)
df = pd.DataFrame(w_rew_3)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_weights_frames.dat', index=False)

w_rew_4 = np.argsort(w_rew_2[:,1])
np.save(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_frames.npy', w_rew_4)
df = pd.DataFrame(w_rew_4)
df.to_csv(f'{BME_DIR}/crossvals/crossval_{tag}/w_rew_sorted_frames.dat', index=False)


## Extract subensemble, by using random.choice, only run once!! without replacement

In [29]:
BME_DIR = "bme_reweight"

weights = np.load(f"{BME_DIR}/crossvals/crossval_{tag}/weights_results.npy")[0]

traj_file = f"gaag_simulations/concat_traj_nopbc.xtc"
top_file = f"gaag_simulations/initial_nopbc_mdtraj.pdb"
pdb_file_out = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
n_frames_out = 100

frames_random = np.random.choice(np.arange(0, len(weights)), size=n_frames_out, replace=False, p=weights)# important modifier: replacement was set to False, instead of True

# Save Frame Clusters as PDB Files
np.save(f"qt_clustering_100/{tag}/subsampled_frames_random_noReplace.npy", frames_random)
traj = md.load(traj_file, top=top_file)
traj_subsampled = traj[frames_random]
traj_subsampled.save_pdb(pdb_file_out)


In [37]:
### weighting of the random subsample
summe = 0
for frame in frames_random:
    val = w_rew_2[frame][1]
    summe = summe + val
print("Weighting of the subensemble:", summe)

summe = 0
for frame in range(20100):
    val = w_rew_2[frame][1]
    summe = summe + val
print("Entire weighting of all frames:", summe)

Weighting of the subensemble: 0.7861186498249826
Entire weighting of all frames: 0.9999999999999988


# Quality Threshold Clustering

In [ ]:
traj_file = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
top_file = f"qt_clustering_100/{tag}/subsample_random_noReplace.pdb"
tag2 = "random_noReplace"

In [4]:
from lib import QT_fk_new as qtc
import barnaba as bb
import bz2
import pickle as cPickle

def save_bz2( outfile, results ):
    with bz2.BZ2File(outfile, 'w' ) as f:
        cPickle.dump(results, f, protocol = 4)

own_matrix_file = f'qt_clustering_100/{tag}/ownmatrix_{tag2}.pbz2'

trajj = md.load(traj_file, top=top_file)

N = trajj.n_frames
ermsd_matrix = np.zeros((N, N), dtype=np.float16)
for cluster_id in range(N):
    if cluster_id % 10 == 0:
        print(f'{cluster_id+10}/{N} frames processed.')
    ermsd_ = bb.ermsd_traj(trajj[cluster_id],trajj[cluster_id:],cutoff=2.4,residues_ref=[5,6,7,8,9,10],residues_target=[5,6,7,8,9,10])
    ermsd_matrix[cluster_id,cluster_id:] = ermsd_
tmp = np.array(ermsd_matrix[:,:]+ermsd_matrix[:,:].T, dtype=np.float32)
save_bz2(own_matrix_file, tmp)


10/100 frames processed.
20/100 frames processed.
30/100 frames processed.
40/100 frames processed.
50/100 frames processed.
60/100 frames processed.
70/100 frames processed.
80/100 frames processed.
90/100 frames processed.
100/100 frames processed.


In [5]:
ermsd_matrix = qtc.load_matrix(own_matrix_file)
cluster_arr = qtc.qt_cluster(ermsd_matrix, cutoff=0.9,minsize=5)
cut = "cutoff_0_9"
np.savetxt(f"qt_clustering_100/{tag}/QT_Clusters_{tag2}_{cut}_minsize5.txt", cluster_arr, fmt="%i")


Precalculated distance matrix provided.
Loading matrix...
Matrix Size 100
>>> Cluster # 1 found with 20 frames at center 51 <<<
>>> Cluster # 2 found with 10 frames at center 53 <<<
>>> Cluster # 3 found with 6 frames at center 71 <<<
>>> Cluster # 4 found with 6 frames at center 86 <<<


### Load Clustering

In [6]:
cluster_array = np.loadtxt(f"qt_clustering_100/{tag}/QT_Clusters_{tag2}_{cut}_minsize5.txt")
cluster_dict = {}
for cluster_id in set(cluster_array):
    cluster_dict[int(cluster_id)] = np.where(cluster_array==cluster_id)[0]
print(cluster_dict)

{0: array([ 3,  4, 14, 18, 25, 26, 37, 38, 40, 51, 55, 58, 62, 64, 65, 69, 73,
       78, 82, 87], dtype=int64), 1: array([ 5,  6,  7,  9, 13, 27, 45, 53, 60, 98], dtype=int64), 2: array([11, 28, 30, 46, 71, 94], dtype=int64), 3: array([20, 35, 48, 61, 81, 86], dtype=int64), -1: array([ 0,  1,  2,  8, 10, 12, 15, 16, 17, 19, 21, 22, 23, 24, 29, 31, 32,
       33, 34, 36, 39, 41, 42, 43, 44, 47, 49, 50, 52, 54, 56, 57, 59, 63,
       66, 67, 68, 70, 72, 74, 75, 76, 77, 79, 80, 83, 84, 85, 88, 89, 90,
       91, 92, 93, 95, 96, 97, 99], dtype=int64)}


### Save Clustered Frames in PDB Files

In [7]:
frames = np.load(f"qt_clustering_100/{tag}/subsampled_frames_{tag2}.npy")
print(frames)

[18833 18320 18320 14388  8247  4132  4166  4136  4781  4083 18893  4132
 18320  4081  1228  3155  4573 18890 13996 15597  4189 18893  4548 18320
 18891  7266 19221  4145  4145 18825  4166 18163 18803 18629 18536  4173
 18163  4238 12684 18809  7943 18803 18809 18891  4238  4171  4081 18163
  4182  4238 18893 11753  7943  4122 18809  2994  4238  4781  2238 18497
  4146  4177 15221  4781 12882 10531 18891  4081 18320 18105  4145  4171
 18893  9515  4622 18320 18809  4238 11250  7020 18320  4171 12665 18893
  4173 18501  4187 12359 18891 18893  7020 18320 19012 18893  4083 18809
  1922  4083  4164  4189]


In [8]:
traj_file = f"gaag_simulations/concat_traj_nopbc.xtc"
top_file = f"gaag_simulations/initial_nopbc_mdtraj.pdb"

traj = md.load(traj_file, top=top_file)
if not os.path.exists(f"qt_clustering_100/{tag}/frames_{tag2}_{cut}"):
    os.makedirs(f"qt_clustering_100/{tag}/frames_{tag2}_{cut}")
for cluster_id  in cluster_dict.keys():
   print(cluster_id)
   id = cluster_dict[cluster_id]
   print(frames[id])
   pdb_file_out = f'qt_clustering_100/{tag}/frames_{tag2}_{cut}/cluster_{cluster_id}.pdb'
   cluster = traj[frames[id]]
   cluster.save_pdb(pdb_file_out)


0
[14388  8247  1228 13996  7266 19221  4238 12684  7943 11753  2994  2238
 15221 12882 10531 18105  9515 11250 12665 12359]
1
[4132 4166 4136 4083 4081 4145 4171 4122 4146 4164]
2
[4132 4145 4166 4081 4171 4083]
3
[4189 4173 4182 4177 4171 4187]
-1
[18833 18320 18320  4781 18893 18320  3155  4573 18890 15597 18893  4548
 18320 18891 18825 18163 18803 18629 18536 18163 18809 18803 18809 18891
  4238 18163  4238 18893  7943 18809  4238  4781 18497  4781 18891  4081
 18320  4145 18893  4622 18320 18809  4238  7020 18320 18893  4173 18501
 18891 18893  7020 18320 19012 18893 18809  1922  4083  4189]


In [9]:
frames_dict = {}
for cluster_id in set(cluster_array):
    id = cluster_dict[cluster_id]
    frames_dict[int(cluster_id)] = frames[id]
print(frames_dict)

{0: array([14388,  8247,  1228, 13996,  7266, 19221,  4238, 12684,  7943,
       11753,  2994,  2238, 15221, 12882, 10531, 18105,  9515, 11250,
       12665, 12359]), 1: array([4132, 4166, 4136, 4083, 4081, 4145, 4171, 4122, 4146, 4164]), 2: array([4132, 4145, 4166, 4081, 4171, 4083]), 3: array([4189, 4173, 4182, 4177, 4171, 4187]), -1: array([18833, 18320, 18320,  4781, 18893, 18320,  3155,  4573, 18890,
       15597, 18893,  4548, 18320, 18891, 18825, 18163, 18803, 18629,
       18536, 18163, 18809, 18803, 18809, 18891,  4238, 18163,  4238,
       18893,  7943, 18809,  4238,  4781, 18497,  4781, 18891,  4081,
       18320,  4145, 18893,  4622, 18320, 18809,  4238,  7020, 18320,
       18893,  4173, 18501, 18891, 18893,  7020, 18320, 19012, 18893,
       18809,  1922,  4083,  4189])}
